# 📈 Linear Regression Interactive Demo  
*A hands-on guide to understanding linear regression with real-time visualization in Google Colab*

---

## 🔍 What is Linear Regression?

**Linear regression** is one of the simplest and most fundamental algorithms in machine learning and statistics. It models the relationship between a **dependent variable** (output, `y`) and one or more **independent variables** (inputs, `X`) by fitting a **linear equation** to observed data.

### 📐 The Mathematical Model

For **simple linear regression** (one input feature), the model is:

$$
\hat{y} = \theta_0 + \theta_1 x
$$

Where:
- $\hat{y}$ = predicted output  
- $x$ = input feature  
- $\theta_0$ = intercept (bias term)  
- $\theta_1$ = slope (weight/coefficient)  

### 🎯 Goal: Minimize Prediction Error

The algorithm finds the best $\theta_0$ and $\theta_1$ by minimizing the **Mean Squared Error (MSE)**:

$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

Where:
- $y_i$ = actual value  
- $\hat{y}_i$ = predicted value  
- $n$ = number of data points  

### ⚙️ How It Works: Gradient Descent

**Gradient descent** is an optimization algorithm that iteratively adjusts the parameters to reduce error:
1. Start with random initial parameters  
2. Compute the gradient (direction of steepest increase in error)  
3. Update parameters in the opposite direction of the gradient  
4. Repeat until convergence  

---

## 🧪 How to Use This Interactive Demo

This notebook provides a **visual, step-by-step demonstration** of linear regression in action.

### 🎮 Interactive Controls

| Button | Function |
|--------|----------|
| **Generate New Data** | Creates a fresh dataset with random points following a linear trend + noise |
| **Gradient Descent Step** | Performs one optimization iteration and updates the regression line |
| **Run All Steps** | Completes all 100 iterations automatically |

### 👁️ What You'll See

1. **Top Plot**:  
   - 🔵 Blue dots: Actual data points  
   - 🔴 Red line: Current regression model  
   - ⚫ Black dashed lines: **Residuals** (prediction errors)  

2. **Bottom Plot**:  
   - 📉 MSE over iterations (shows how error decreases)  

3. **Log Output**:  
   - Current MSE value after each step  

### 💡 Key Learning Points

- **Residuals shrink** as the model improves  
- **MSE decreases** with each optimization step  
- **Different datasets** produce different optimal lines  
- **Random initialization** affects the starting point but not the final result  

## Setup all code libraries

In [1]:
# @title
# Cell 1: Setup — Pure data and algorithm logic
# !pip install ipywidgets matplotlib numpy

In [2]:
# @title
# Linear Regression with Error Visualization — Fully Working in Google Colab
# !pip install ipywidgets matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Button, Output, VBox, HBox, HTML
from IPython.display import display, clear_output

# ===== GLOBAL STATE =====
X = None
y = None
theta = None
mse_history = []
iteration = 0
learning_rate = 0.001  # Stable for raw data
max_iterations = 100

# ===== OUTPUT AREAS =====
plot_out = Output()
mse_out = Output()
log_out = Output()
theta_out = Output()  # New: for theta display

# ===== CORE FUNCTIONS =====
def generate_data():
    global X, y, theta, mse_history, iteration
    n = 50
    noise = 10
    X_raw = np.random.randn(n) * 10
    y_true = 2.5 * X_raw + 3
    y = y_true + np.random.randn(n) * noise
    X = np.column_stack((np.ones(n), X_raw))

    theta = np.random.randn(2) * 0.01
    mse_history = []
    iteration = 0

    y_pred = X.dot(theta)
    mse = np.mean((y - y_pred)**2)
    mse_history.append(mse)
    update_theta_display()

def gradient_step():
    global theta, mse_history, iteration, X, y
    if X is None:
        return "⚠️ Generate data first!"
    if iteration >= max_iterations:
        return "✅ Done!"

    y_pred_old = X.dot(theta)
    m = len(y)
    d0 = (-2/m) * np.sum(y - y_pred_old)
    d1 = (-2/m) * np.sum((y - y_pred_old) * X[:,1])

    theta[0] -= learning_rate * d0
    theta[1] -= learning_rate * d1

    y_pred_new = X.dot(theta)
    new_mse = np.mean((y - y_pred_new)**2)
    mse_history.append(new_mse)
    iteration += 1

    update_theta_display()
    return f"Step {iteration}: MSE = {new_mse:.6f}"

def update_theta_display():
    """Update the theta display widget"""
    with theta_out:
        clear_output(wait=True)
        if theta is not None:
            print(f"θ₀ (intercept) = {theta[0]:.4f}    |    θ₁ (slope) = {theta[1]:.4f}")
        else:
            print("θ₀ = ?    |    θ₁ = ?")

# ===== PLOTTING WITH ERROR LINES =====
def update_plots():
    with plot_out:
        clear_output(wait=True)
        plt.figure(figsize=(10, 5))
        if X is not None:
            x_vals = X[:, 1]
            y_pred = X.dot(theta)

            # Plot data points
            plt.scatter(x_vals, y, alpha=0.7, label='Data points', color='steelblue')

            # Plot regression line
            x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
            y_line = theta[0] + theta[1] * x_line
            plt.plot(x_line, y_line, 'r-', lw=2, label='Regression line')

            # 🔸 Draw error lines (residuals)
            for i in range(len(x_vals)):
                plt.plot([x_vals[i], x_vals[i]], [y[i], y_pred[i]],
                        'k--', alpha=0.4, linewidth=0.8)

            plt.title(f'Linear Regression with Errors (MSE: {mse_history[-1]:.4f})')
            plt.xlabel('X')
            plt.ylabel('y')
            plt.legend()
            plt.grid(alpha=0.3)
            plt.tight_layout()
        else:
            plt.text(0.5, 0.5, 'No data', ha='center', va='center')
            plt.title('Linear Regression with Errors')
        plt.show()

    with mse_out:
        clear_output(wait=True)
        plt.figure(figsize=(10, 2.5))
        if mse_history:
            plt.plot(mse_history, 'b-o', markersize=3)
            plt.title('MSE Over Iterations')
            plt.xlabel('Iteration')
            plt.ylabel('MSE')
            plt.grid(alpha=0.3)
            plt.tight_layout()
        else:
            plt.text(0.5, 0.5, 'No training', ha='center', va='center')
            plt.title('MSE Over Iterations')
        plt.show()

# ===== BUTTONS =====
def on_generate(_):
    with log_out:
        clear_output()
        generate_data()
        update_plots()

def on_step(_):
    with log_out:
        clear_output(wait=True)
        msg = gradient_step()
        print(msg)
        if "⚠️" not in msg:
            update_plots()

def on_run_all(_):
    with log_out:
        clear_output(wait=True)
        while iteration < max_iterations and X is not None:
            gradient_step()
        print("✅ Optimization complete!")
        update_plots()

btn_generate = Button(description="Generate New Data", button_style='success')
btn_step = Button(description="Gradient Descent Step", button_style='info')
btn_run = Button(description="Run All Steps", button_style='warning')

btn_generate.on_click(on_generate)
btn_step.on_click(on_step)
btn_run.on_click(on_run_all)

# Initialize
update_plots()

display(VBox([
    HTML("<h2>Linear Regression with Error Lines</h2>"),
    HTML("<p>Vertical dashed lines show residuals (error = actual − predicted)</p>"),
    HBox([btn_generate, btn_step, btn_run]),
    HTML("<h3>Current Parameters:</h3>"),
    theta_out,
    log_out,
    plot_out,
    mse_out
]))